# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshitha922/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Random Forest model and its probability scores to rank pages for content refresh review. This fits my lane because I want to identify which pages should be reviewed first. I will compare its ranking performance with my Week-4 baseline using the same split and metric.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
method = "Random Forest"
metric = "Precision@50"

print("Selected method:", method)
print("Evaluation metric:", metric)

Selected method: Random Forest
Evaluation metric: Precision@50


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-based split so that pages from the same client do not appear in both training and testing data. This helps check whether the model can work on a client it has not seen before, making the evaluation more honest.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load the starter dataset
data_url = "https://raw.githubusercontent.com/varshitha922/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_url)

# Keep each client's pages together
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Check that no client appears in both sets
shared_clients = set(train_df["client_id"]) & set(test_df["client_id"])

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Clients in training:", train_df["client_id"].nunique())
print("Clients in testing:", test_df["client_id"].nunique())
print("Shared clients:", len(shared_clients))

Training rows: 23837
Testing rows: 6163
Clients in training: 25
Clients in testing: 7
Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Create the target: 1 means the page has a declining trend.
y_train = train_df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
y_test = test_df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

# Use numeric signals only. Exclude IDs and target/leakage-related fields.
excluded_words = [
    "id", "trend", "declin", "label", "future",
    "flag", "action", "score", "rank", "target",
    "last_30d", "prev_30d"
]

feature_cols = [
    col for col in train_df.select_dtypes(include=np.number).columns
    if not any(word in col.lower() for word in excluded_words)
    and col != "impressions_90d"
]

print("Number of model features:", len(feature_cols))

# Train Random Forest on the training clients.
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
)

model.fit(train_df[feature_cols], y_train)

# Get model probability scores for the held-out test clients.
model_scores = model.predict_proba(test_df[feature_cols])[:, 1]

# Recreate the W04 baseline rule on the same test clients.
baseline_scores = np.where(
    test_df["impressions_90d"] >= 500,
    test_df["impressions_90d"],
    0
)

# Calculate Precision@50.
def precision_at_50(y_true, scores):
    top_indices = np.argsort(-np.asarray(scores), kind="mergesort")[:50]
    return np.asarray(y_true)[top_indices].mean()

baseline_p50 = precision_at_50(y_test, baseline_scores)
model_p50 = precision_at_50(y_test, model_scores)
base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": ["W04 visibility rule", "Random Forest", "Test-set base rate"],
    "Precision@50": [baseline_p50, model_p50, base_rate]
})

display(comparison)


Number of model features: 22


,Method,Precision@50
0,W04 visibility rule,0.440000
1,Random Forest,0.580000
2,Test-set base rate,0.510952


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The three most important features were avg_position, days_with_impressions, and content_age_days. This shows that the model relied on search position, the number of days with impressions, and content age when making predictions.

The model made 2,638 incorrect predictions using a 0.5 cutoff. Some examples were pages with an actual label of 1 that the model predicted as 0, while others had an actual label of 0 that the model predicted as 1. This shows that the model can make mistakes in both directions. Its scores should be used for decision support, not as final decisions about which pages need a content refresh.



In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the 3 most important features used by the Random Forest.
rf = model.named_steps["randomforestclassifier"]

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 3 important features:")
display(importance.head(3))

# Find cases where the model's 0.5 decision differs from the actual label.
predicted = (model_scores >= 0.5).astype(int)
actual = np.asarray(y_test)

error_positions = np.where(predicted != actual)[0]

print("Number of incorrect predictions:", len(error_positions))

# Display up to 3 errors without showing client or content IDs.
error_rows = []
for pos in error_positions[:3]:
    row = {
        "Test row position": int(pos),
        "Actual label": int(actual[pos]),
        "Predicted label": int(predicted[pos]),
        "Model score": round(float(model_scores[pos]), 3)
    }
    for feature in importance.head(3)["Feature"]:
        row[feature] = test_df.iloc[pos][feature]
    error_rows.append(row)

print("Examples of incorrect predictions:")
display(pd.DataFrame(error_rows))

Top 3 important features:


,Feature,Importance
18,avg_position,0.127932
12,days_with_impressions,0.115128
14,content_age_days,0.088795


Number of incorrect predictions: 2638
Examples of incorrect predictions:


,Test row position,Actual label,Predicted label,Model score,avg_position,days_with_impressions,content_age_days
0,1,1,0,0.49,20.3,88,445
1,3,0,1,0.80,39.8,69,238
2,7,1,0,0.36,13.9,43,502


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.